# Laboratorio #9 - Embeddings

**Juan Diego Letona Aguilar**

*20230285*

In [32]:
# Notebook 9 (el pdf dice laboratorio 8, pero corresponde al laboratorio 9).
# la primera parte (imports, carga de paquetes, etc.) es parecida a las iteraciones anteriores
# principalmente librerias y carga de paquetes necesarios para el proyecto

import sys
import subprocess

packages = [
    "pandas",
    "numpy",
    "nltk",
    "matplotlib",
    "scikit-learn",
    "scipy",
    "gensim",
    "torch",
    "tqdm"
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", *packages
])

0

In [33]:
from pathlib import Path
from collections import Counter
import re
import time
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import sent_tokenize
from tqdm.auto import tqdm

from gensim.models import KeyedVectors, Word2Vec, FastText

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

#semilla para que los resultados sean reproducibles
SEED = 42
np.random.seed(SEED)

#recursos de NLTK
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 1. Preparación de los corpus

**Carga de los corpus**

In [34]:
#cargamos el corpus de noticias (mismo dataset y misma limpieza basica de los laboratorios 1 y 2)

df = pd.read_csv("df_total.csv")

URL_COL = "url"
TEXT_COL = "news"
CATEGORY_COL = "Type"

corpus = df[[URL_COL, TEXT_COL, CATEGORY_COL]].copy()

corpus[TEXT_COL] = corpus[TEXT_COL].fillna("").astype(str)
corpus = corpus[corpus[TEXT_COL].str.strip().ne("")]
corpus = corpus.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

print("Dimensiones del DataFrame original:", df.shape)
print("Documentos de noticias despues de quitar vacios y duplicados:", len(corpus))

Dimensiones del DataFrame original: (1217, 3)
Documentos de noticias despues de quitar vacios y duplicados: 1137


In [35]:
#cargamos el Quijote (texto de Project Gutenberg)

with open("don-quijote.txt", encoding="utf-8-sig") as archivo:
    texto_quijote = archivo.read()

#quitamos la licencia y el encabezado de gutenberg, que no forman parte de la obra
inicio = texto_quijote.find("*** START")
fin = texto_quijote.find("*** END")
texto_quijote = texto_quijote[inicio:fin]
texto_quijote = texto_quijote.split("\n", 1)[1]

#cada capitulo es un documento
documentos_quijote = re.split(r"(?m)^\s*Cap[ií]tulo\s+", texto_quijote)
documentos_quijote = [doc for doc in documentos_quijote if doc.strip()]

print("Caracteres del Quijote:", f"{len(texto_quijote):,}")
print("Documentos del Quijote (capitulos y preliminares):", len(documentos_quijote))
print(documentos_quijote[1][:300])

Caracteres del Quijote: 2,098,169
Documentos del Quijote (capitulos y preliminares): 127
primero. Que trata de la condición y ejercicio del famoso hidalgo
don Quijote de la Mancha


En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho
tiempo que vivía un hidalgo de los de lanza en astillero, adarga antigua,
rocín flaco y galgo corredor. Una olla de algo más vaca que


**Segmentación en oraciones y tokenización**

Primero se arreglan algunos detalles del texto crudo. En las noticias hay lugares donde falta el espacio después del punto (por ejemplo "imaginamos.Para"), lo cual confunde al segmentador. En el Quijote el texto viene cortado en renglones, así que se unen antes de segmentar.

In [36]:
#funciones para segmentar en oraciones y tokenizar en palabras

patron_palabras = re.compile(r"[a-záéíóúüñ]+")

def segmentar_oraciones(texto):
    #agregamos el espacio que falta despues de un punto pegado a una mayuscula
    texto = re.sub(r"([a-záéíóúñ])([.!?])([A-ZÁÉÍÓÚÑ¿¡])", r"\1\2 \3", texto)
    #unimos los renglones y los espacios multiples
    texto = re.sub(r"\s+", " ", texto).strip()
    return sent_tokenize(texto, language="spanish")

def tokenizar(oracion):
    #minusculas y solo palabras (letras con tilde, ñ y ü), sin puntuacion ni numeros
    return patron_palabras.findall(oracion.lower())

def preparar_corpus(documentos):
    #devuelve una lista de listas de tokens (una lista por oracion) y el numero de oraciones por documento
    oraciones_tokens = []
    oraciones_por_doc = []

    for documento in documentos:
        tokens_doc = [tokenizar(oracion) for oracion in segmentar_oraciones(documento)]
        #descartamos oraciones con menos de 2 tokens porque no tienen contexto
        tokens_doc = [oracion for oracion in tokens_doc if len(oracion) >= 2]
        oraciones_tokens.extend(tokens_doc)
        oraciones_por_doc.append(len(tokens_doc))

    return oraciones_tokens, oraciones_por_doc

oraciones_noticias, oraciones_por_doc_noticias = preparar_corpus(
    tqdm(corpus[TEXT_COL], desc="Noticias")
)
oraciones_quijote, oraciones_por_doc_quijote = preparar_corpus(
    tqdm(documentos_quijote, desc="Quijote")
)

print("Ejemplo de oracion de noticias:")
print(oraciones_noticias[10])
print("\nEjemplo de oracion del Quijote:")
print(oraciones_quijote[10])

Quijote: 100%|██████████| 127/127 [00:00<00:00, 498.67it/s]

Ejemplo de oracion de noticias:
['la', 'comisión', 'reguladora', 'de', 'valores', 'de', 'china', 'csrc', 'dijo', 'en', 'un', 'comunicado', 'que', 'tomó', 'nota', 'de', 'los', 'nuevos', 'requisitos', 'de', 'la', 'comisión', 'de', 'valores', 'de', 'estados', 'unidos', 'sec', 'para', 'las', 'empresas', 'chinas', 'y', 'que', 'ambas', 'partes', 'deberían', 'mantener', 'el', 'espíritu', 'de', 'respeto', 'mutuo', 'y', 'reforzar', 'las', 'comunicaciones', 'sobre', 'la', 'regulación']

Ejemplo de oracion del Quijote:
['la', 'cual', 'dicha', 'pena', 'sea', 'la', 'tercia', 'parte', 'para', 'la', 'persona', 'que', 'lo', 'acusare', 'y', 'la', 'otra', 'tercia', 'parte', 'para', 'nuestra', 'cámara', 'y', 'la', 'otra', 'tercia', 'parte', 'para', 'el', 'juez', 'que', 'lo', 'sentenciare']


**Normalización: decisiones y compatibilidad con fastText SUC**

In [37]:
#cargamos los vectores preentrenados de fastText SUC (tamaño M, 100 dimensiones)
#solo las 200,000 palabras mas frecuentes para no llenar la memoria

RUTA_SUC = Path("embeddings-m-model.vec")
URL_SUC = "https://zenodo.org/record/3234051/files/embeddings-m-model.vec?download=1"

if not RUTA_SUC.exists():
    print("Descargando embeddings-m-model.vec (1.1 GB), esto tarda un rato...")
    urllib.request.urlretrieve(URL_SUC, RUTA_SUC)

suc = KeyedVectors.load_word2vec_format(str(RUTA_SUC), limit=200_000)

print("Palabras en suc:", len(suc.key_to_index))
print("Dimensiones:", suc.vector_size)

Palabras en suc: 200000
Dimensiones: 100


In [38]:
#revisamos si suc distingue mayusculas y tildes

pruebas = ["españa", "España", "ESPAÑA", "espana", "méxico", "mexico", "Mexico", "también", "tambien", "gobierno", "Gobierno"]

revision_suc = pd.DataFrame({
    "palabra": pruebas,
    "esta_en_suc": [palabra in suc.key_to_index for palabra in pruebas],
    "posicion_en_vocabulario": [suc.key_to_index.get(palabra) for palabra in pruebas]
})

display(
    revision_suc.style
    .hide(axis="index")
    .set_caption("Revisión del vocabulario de SUC")
)

#cuantos tokens de cada corpus (ya normalizados) existen en suc
for nombre, oraciones in [("Noticias", oraciones_noticias), ("Quijote", oraciones_quijote)]:
    tokens = [token for oracion in oraciones for token in oracion]
    cobertura = np.mean([token in suc.key_to_index for token in tokens]) * 100
    print(f"{nombre}: {cobertura:.2f}% de los tokens estan en suc")

palabra,esta_en_suc,posicion_en_vocabulario
españa,True,541.000000
España,False,nan
ESPAÑA,False,nan
espana,True,67448.000000
méxico,True,797.000000
mexico,True,16941.000000
Mexico,False,nan
también,True,64.000000
tambien,True,4259.000000
gobierno,True,236.000000


Noticias: 98.79% de los tokens estan en suc
Quijote: 97.38% de los tokens estan en suc


## Decisiones de normalización

- Minúsculas: sí. SUC está completamente en minúsculas: "españa" está en el vocabulario (posición 541), pero "España" y "ESPAÑA" no. Si dejamos las mayúsculas, nuestras palabras no se podrían buscar en SUC y, además, "Gobierno" y "gobierno" serían dos tipos distintos con la mitad de contextos cada uno.
- Tildes: se conservan. SUC sí las distingue: "méxico" está en la posición 797 y "mexico" en la 16,941, es decir, la versión sin tilde es una variante poco frecuente (casi un error de escritura). Quitar tildes nos dejaría con formas poco comunes en SUC y, además, mezclaría palabras distintas ("él"/"el", "sí"/"si", "más"/"mas").
- Puntuación y números: se eliminan. Con la expresión regular solo quedan palabras (letras, tildes, ñ y ü). Los números aportan poco a los vecinos semánticos y llenarían el vocabulario de tipos muy raros.
- Compatibilidad con SUC: con esta normalización, el 98.79 % de los tokens de las noticias y el 97.38 % de los del Quijote existen en SUC, por lo que la comparación de las secciones 5 y 8 es justa.

## ¿Vale la pena quitar stopwords o lematizar antes de entrenar embeddings?

En este caso, no.

- Stopwords: Word2Vec cuenta posiciones, no palabras "importantes". Si quitamos "de", "la" o "que", palabras que estaban separadas por dos o tres posiciones quedan juntas y la ventana de contexto se deforma. Además, las stopwords dan información sintáctica (qué tipo de palabra suele ir antes o después) y Word2Vec ya trae el parámetro `sample` para submuestrear las palabras muy frecuentes sin destruir la oración.
- Lematización: en los laboratorios 1 y 2 servía para reducir el vocabulario en BoW y TF-IDF. Aquí perdería información porque si "gobiernos" y "gobernó" se colapsan en una sola forma, se pierde la diferencia entre sustantivo y verbo, entre plural y singular y entre pasado y presente, y el modelo ya no podría aprender que "gobernó" se parece a otros verbos en pasado. Y SUC no está lematizado, así que perderíamos compatibilidad.

**Tabla de estadísticas por corpus**

In [39]:
#documentos, oraciones, tokens y tipos de cada corpus

def estadisticas_corpus(nombre, n_documentos, oraciones):
    tokens = [token for oracion in oraciones for token in oracion]
    return {
        "corpus": nombre,
        "documentos": n_documentos,
        "oraciones": len(oraciones),
        "tokens": len(tokens),
        "tipos": len(set(tokens))
    }

tabla_corpus = pd.DataFrame([
    estadisticas_corpus("Noticias", len(corpus), oraciones_noticias),
    estadisticas_corpus("Don Quijote", len(documentos_quijote), oraciones_quijote)
])

display(
    tabla_corpus.style
    .hide(axis="index")
    .format({"documentos": "{:,}", "oraciones": "{:,}", "tokens": "{:,}", "tipos": "{:,}"})
    .set_caption("Tamaño de cada corpus")
)

corpus,documentos,oraciones,tokens,tipos
Noticias,"1,137","19,939","593,044","27,762"
Don Quijote,127,"9,399","380,962","22,882"


**Vocabulario con distintos valores de frecuencia mínima**

In [40]:
#vocabulario para min_count = 1, 2 y 5 en el corpus de noticias

frecuencias_noticias = Counter(token for oracion in oraciones_noticias for token in oracion)
total_tokens_noticias = sum(frecuencias_noticias.values())

resultados_min_count = []

for min_count in [1, 2, 5]:
    vocabulario = {palabra for palabra, f in frecuencias_noticias.items() if f >= min_count}
    tokens_fuera = sum(f for palabra, f in frecuencias_noticias.items() if f < min_count)

    resultados_min_count.append({
        "min_count": min_count,
        "tamano_vocabulario": len(vocabulario),
        "tokens_fuera": tokens_fuera,
        "tokens_fuera_%": tokens_fuera / total_tokens_noticias * 100
    })

tabla_min_count = pd.DataFrame(resultados_min_count)

display(
    tabla_min_count.style
    .hide(axis="index")
    .format({"tamano_vocabulario": "{:,}", "tokens_fuera": "{:,}", "tokens_fuera_%": "{:.2f}%"})
    .set_caption("Vocabulario del corpus de noticias según min_count")
)

min_count,tamano_vocabulario,tokens_fuera,tokens_fuera_%
1,"27,762",0,0.00%
2,"15,813","11,949",2.01%
5,"8,278","32,013",5.40%


Con min_count = 1, el vocabulario tiene 27,762 tipos, pero 11,949 de ellos (43%) aparecen una sola vez y solo representan el 2.01% de los tokens. Con min_count = 2, el vocabulario baja a 15,813 tipos, una reduccion del 43%, y con min_count = 5 baja a 8,278 tipos, una reduccion del 70%, perdiendo apenas el 5.40% de los tokens. Esto significa que el vocabulario es muy grande debido a la cantidad de palabras raras, siguiendo la distribucion de Zipf, mientras que el texto casi no se ve afectado. Una palabra que aparece una sola vez no tiene suficientes contextos para aprender un vector confiable. Por eso, min_count = 2 es un buen equilibrio, ya que conserva el 98% del texto y elimina casi la mitad de los tipos.

**Pares (palabra central, contexto) de skip-gram hechos a mano**

In [41]:
#generamos a mano los pares de skip-gram para una oracion de noticias

def generar_pares_skipgram(oracion, ventana):
    pares = []
    for i, central in enumerate(oracion):
        inicio = max(0, i - ventana)
        fin = min(len(oracion), i + ventana + 1)
        for j in range(inicio, fin):
            if j != i:
                pares.append((central, oracion[j]))
    return pares

#buscamos una oracion corta para que se pueda ver completa
oracion_ejemplo = next(oracion for oracion in oraciones_noticias if 7 <= len(oracion) <= 8)
pares_ejemplo = generar_pares_skipgram(oracion_ejemplo, ventana=2)

print("Oracion:", " ".join(oracion_ejemplo))
print("Tokens:", len(oracion_ejemplo), "| Pares con ventana 2:", len(pares_ejemplo))

#mostramos los pares agrupados por palabra central, como en la diapositiva de la clase
for central in oracion_ejemplo:
    contextos = [c for (w, c) in pares_ejemplo if w == central]
    print(f"({central}) -> {contextos}")

Oracion: viva aerobus es como un hijo para mí
Tokens: 8 | Pares con ventana 2: 26
(viva) -> ['aerobus', 'es']
(aerobus) -> ['viva', 'es', 'como']
(es) -> ['viva', 'aerobus', 'como', 'un']
(como) -> ['aerobus', 'es', 'un', 'hijo']
(un) -> ['es', 'como', 'hijo', 'para']
(hijo) -> ['como', 'un', 'para', 'mí']
(para) -> ['un', 'hijo', 'mí']
(mí) -> ['hijo', 'para']


**Cantidad de pares de entrenamiento del corpus completo**

In [42]:
#numero de pares para ventana 2 y ventana 5 sin generarlos todos
#para una oracion de n tokens, la palabra en la posicion i tiene min(i, w) vecinos a la izquierda y min(n-1-i, w) a la derecha

def contar_pares(oraciones, ventana):
    total = 0
    for oracion in oraciones:
        n = len(oracion)
        posiciones = np.arange(n)
        total += int((np.minimum(posiciones, ventana) + np.minimum(n - 1 - posiciones, ventana)).sum())
    return total

#verificamos la formula con el generador manual
assert contar_pares([oracion_ejemplo], 2) == len(pares_ejemplo)

tabla_pares = pd.DataFrame([
    {
        "ventana": ventana,
        "pares": contar_pares(oraciones_noticias, ventana),
        "tokens": total_tokens_noticias
    }
    for ventana in [2, 5]
])
tabla_pares["pares_por_token"] = tabla_pares["pares"] / tabla_pares["tokens"]

display(
    tabla_pares.style
    .hide(axis="index")
    .format({"pares": "{:,}", "tokens": "{:,}", "pares_por_token": "{:.2f}"})
    .set_caption("Pares de entrenamiento de skip-gram en el corpus de noticias")
)

ventana,pares,tokens,pares_por_token
2,"2,252,542","593,044",3.80
5,"5,333,292","593,044",8.99


Con ventana 2, el corpus de noticias produce 2,252,542 pares (3.80 por token), y con ventana 5 produce 5,333,292 pares (8.99 por token). Son menos de 4 y 10 pares por token porque las palabras que estan en los bordes de la oracion tienen menos vecinos. Con apenas 593,044 tokens se generan millones de ejemplos de entrenamiento, y estos crecen casi linealmente con el tamano de la ventana.

Word2Vec no necesita datos etiquetados porque la etiqueta se obtiene del mismo texto. La tarea consiste en predecir una palabra vecina a partir de la palabra central, y la respuesta correcta ya esta escrita en la oracion. Es un tipo de aprendizaje autosupervisado, por lo que no hace falta que una persona anote los datos. Cualquier texto sin etiquetar puede generar sus propios pares de entrenamiento (palabra central y contexto).

## 2. Co-ocurrencia, PPMI y SVD

**Matriz de co-ocurrencia palabra × palabra (ventana ±2, 2,000 palabras más frecuentes)**

In [43]:
#vocabulario de las 2,000 palabras mas frecuentes del corpus de noticias

N_PALABRAS = 2000
VENTANA_COOC = 2

palabras_cooc = [palabra for palabra, _ in frecuencias_noticias.most_common(N_PALABRAS)]
indice_cooc = {palabra: i for i, palabra in enumerate(palabras_cooc)}

#convertimos todo el corpus a indices (-1 para palabras fuera del top 2000)
#y guardamos a que oracion pertenece cada token para no cruzar los limites de la oracion
ids = np.array([indice_cooc.get(token, -1) for oracion in oraciones_noticias for token in oracion])
ids_oracion = np.repeat(np.arange(len(oraciones_noticias)), [len(oracion) for oracion in oraciones_noticias])

#matriz de co-ocurrencia: para cada distancia d sumamos los pares (i, i+d) de la misma oracion
matriz_cooc = np.zeros((N_PALABRAS, N_PALABRAS), dtype=np.float64)

for d in range(1, VENTANA_COOC + 1):
    mismo_contexto = (ids_oracion[:-d] == ids_oracion[d:]) & (ids[:-d] >= 0) & (ids[d:] >= 0)
    izquierda = ids[:-d][mismo_contexto]
    derecha = ids[d:][mismo_contexto]
    np.add.at(matriz_cooc, (izquierda, derecha), 1)
    np.add.at(matriz_cooc, (derecha, izquierda), 1)

total_celdas = matriz_cooc.size
celdas_cero = int((matriz_cooc == 0).sum())
dispersion_cooc = celdas_cero / total_celdas * 100

print("Forma de la matriz de co-ocurrencia:", matriz_cooc.shape)
print("Total de celdas:", f"{total_celdas:,}")
print("Celdas con valor cero:", f"{celdas_cero:,}")
print(f"Porcentaje de ceros: {dispersion_cooc:.2f}%")
print("Co-ocurrencias totales (cada par cuenta dos veces):", f"{int(matriz_cooc.sum()):,}")

Forma de la matriz de co-ocurrencia: (2000, 2000)
Total de celdas: 4,000,000
Celdas con valor cero: 3,753,448
Porcentaje de ceros: 93.84%
Co-ocurrencias totales (cada par cuenta dos veces): 1,534,452


In [44]:
#comparamos con la dispersion de la matriz BoW (documentos x palabras) como en el laboratorio 2
from sklearn.feature_extraction.text import CountVectorizer

#reconstruimos los documentos de noticias como listas de tokens
limites = np.cumsum([0] + oraciones_por_doc_noticias)
documentos_tokens = [
    [token for oracion in oraciones_noticias[limites[i]:limites[i + 1]] for token in oracion]
    for i in range(len(oraciones_por_doc_noticias))
]

vectorizador_bow = CountVectorizer(analyzer=lambda doc: doc)
matriz_bow = vectorizador_bow.fit_transform(documentos_tokens)

dispersion_bow = (1 - matriz_bow.nnz / (matriz_bow.shape[0] * matriz_bow.shape[1])) * 100

comparacion_dispersion = pd.DataFrame({
    "matriz": ["Co-ocurrencia (2,000 x 2,000)", "BoW de este notebook (sin lematizar)", "BoW del laboratorio 2 (lematizado)"],
    "forma": [str(matriz_cooc.shape), str(matriz_bow.shape), "(1137, 27799)"],
    "ceros_%": [dispersion_cooc, dispersion_bow, 99.30]
})

display(
    comparacion_dispersion.style
    .hide(axis="index")
    .format({"ceros_%": "{:.2f}%"})
    .set_caption("Dispersión de las matrices")
)

matriz,forma,ceros_%
"Co-ocurrencia (2,000 x 2,000)","(2000, 2000)",93.84%
BoW de este notebook (sin lematizar),"(1137, 27762)",99.14%
BoW del laboratorio 2 (lematizado),"(1137, 27799)",99.30%


La matriz de co-ocurrencia tiene una forma de 2,000 x 2,000 y un 93.84% de ceros, mientras que la matriz BoW tiene un 99.14% de ceros (99.30% en el laboratorio 2, con lematizacion). Ambas son dispersas, pero la matriz de co-ocurrencia es bastante mas densa. Esto se debe a que cada fila corresponde a una palabra frecuente que aparece en muchos contextos distintos, mientras que en BoW cada documento utiliza solo una pequena parte de las mas de 27 mil palabras del vocabulario. Ademas, la matriz de co-ocurrencia es cuadrada y su tamano no depende del numero de documentos.